In [1]:
import pandas as pd

import os

# Set the correct path to your data folder
data_folder = r"C:\Users\linda\OneDrive\Documents\2025-YEAR 3\Semester 2\CMPG 321\PROJECT\OLDZ data"

files = [
    "Age Analysis.xlsx",
    "Customer Account Parameters.xlsx",
    "Customer Categories.xlsx", 
    "Customer Regions.xlsx",
    "Customer.xlsx",
    "Payment Header.xlsx",
    "Payment Lines.xlsx", 
    "Product Brands.xlsx",
    "Product Categories.xlsx",
    "Product Ranges.xlsx",
    "Products Styles.xlsx",
    "Products.xlsx",
    "Purchases Headers.xlsx",
    "Purchases Lines.xlsx",
    "Representatives.xlsx", 
    "Sales Line.xlsx",
    "Sales Header.xlsx",
    "Suppliers.xlsx",
    "Trans Types.xlsx"
]

# Load files from the correct folder
dfs = {}
for f in files:
    file_path = os.path.join(data_folder, f)
    try:
        dfs[f.split(".")[0]] = pd.read_excel(file_path)
        print(f"✓ Loaded {f}")
    except Exception as e:
        print(f"✗ Failed to load {f}: {e}")

print(f"\nSuccessfully loaded {len(dfs)} files")

✓ Loaded Age Analysis.xlsx
✓ Loaded Customer Account Parameters.xlsx
✓ Loaded Customer Categories.xlsx
✓ Loaded Customer Regions.xlsx
✓ Loaded Customer.xlsx
✓ Loaded Payment Header.xlsx
✓ Loaded Payment Lines.xlsx
✓ Loaded Product Brands.xlsx
✓ Loaded Product Categories.xlsx
✓ Loaded Product Ranges.xlsx
✓ Loaded Products Styles.xlsx
✓ Loaded Products.xlsx
✓ Loaded Purchases Headers.xlsx
✓ Loaded Purchases Lines.xlsx
✓ Loaded Representatives.xlsx
✓ Loaded Sales Line.xlsx
✓ Loaded Sales Header.xlsx
✓ Loaded Suppliers.xlsx
✓ Loaded Trans Types.xlsx

Successfully loaded 19 files


In [ ]:
print(dfs.keys())

dict_keys(['Age Analysis', 'Customer Account Parameters', 'Customer Categories', 'Customer Regions', 'Customer', 'Payment Header', 'Payment Lines', 'Product Brands', 'Product Categories', 'Product Ranges', 'Products Styles', 'Products', 'Purchases Headers', 'Purchases Lines', 'Representatives', 'Sales Line', 'Sales Header', 'Suppliers', 'Trans Types'])


In [2]:
clean_dfs = {}

for name,df in dfs.items():
    df.columns= (df.columns.str.strip().str.replace(" ","_").str.replace(".","_"))
    df = df.where(pd.notnull(df), None)
    df = df.convert_dtypes()
    clean_dfs[name]=df



In [3]:
# CHECK WHAT COLLECTIONS WE HAVE
print("Collections in clean_dfs:")
for name in clean_dfs.keys():
    print(f"  - '{name}'")

# Then run your financial period code...

Collections in clean_dfs:
  - 'Age Analysis'
  - 'Customer Account Parameters'
  - 'Customer Categories'
  - 'Customer Regions'
  - 'Customer'
  - 'Payment Header'
  - 'Payment Lines'
  - 'Product Brands'
  - 'Product Categories'
  - 'Product Ranges'
  - 'Products Styles'
  - 'Products'
  - 'Purchases Headers'
  - 'Purchases Lines'
  - 'Representatives'
  - 'Sales Line'
  - 'Sales Header'
  - 'Suppliers'
  - 'Trans Types'


In [4]:

# ADD FINANCIAL PERIOD TO SALES DATA
from datetime import datetime, timedelta

def calculate_financial_period(date_str):
    """Calculate financial period for ClearVue"""
    try:
        if isinstance(date_str, str):
            date = datetime.strptime(date_str, '%Y-%m-%d %H:%M:%S')
        else:
            date = date_str
            
        return f"{date.year}-M{date.month:02d}"
    except:
        return "Unknown"

# Apply to Sales Header
if "Sales Header" in clean_dfs:
    clean_dfs["Sales Header"]['financial_period'] = clean_dfs["Sales Header"]['TRANS_DATE'].apply(calculate_financial_period)
    print("✓ Added financial_period to Sales Header")
    print(f"Sample: {clean_dfs['Sales Header']['financial_period'].iloc[0]}")
else:
    print("Sales Header not found in clean_dfs")
    print("Available collections:", list(clean_dfs.keys()))

✓ Added financial_period to Sales Header
Sample: 2019-M03


In [ ]:
df.duplicated().sum()

np.int64(0)

In [5]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["DB1"]

for name,df in clean_dfs.items():
    records = df.to_dict(orient="records")
    db[name].insert_many(records)
    

In [6]:
# VERIFY FINANCIAL PERIOD IS IN MONGODB
print("Checking financial period in new database...")

# Check Sales Header collection
sales_sample = db["Sales Header"].find_one({"financial_period": {"$exists": True}})
if sales_sample:
    print("✓ SUCCESS: financial_period found in MongoDB!")
    print(f"  Sample: {sales_sample['TRANS_DATE']} -> {sales_sample['financial_period']}")
else:
    print("✗ financial_period NOT found in MongoDB")

# Check how many records have financial_period
count_with_period = db["Sales Header"].count_documents({"financial_period": {"$exists": True}})
total_records = db["Sales Header"].count_documents({})
print(f"  Records with financial_period: {count_with_period}/{total_records}")

# Show some different periods
periods = db["Sales Header"].distinct("financial_period")
print(f"  Unique financial periods: {periods[:5]}...")  # Show first 5

Checking financial period in new database...
✓ SUCCESS: financial_period found in MongoDB!
  Sample: 2019-03-25 00:00:00 -> 2019-M03
  Records with financial_period: 67988/67988
  Unique financial periods: ['2015-M03', '2015-M04', '2015-M05', '2015-M06', '2015-M08']...


In [7]:
# CREATE INDEX FOR FINANCIAL PERIOD
db["Sales Header"].create_index([("financial_period", 1)])
print("✓ Created index on financial_period field")

✓ Created index on financial_period field
